# How machines learn: a colour clustering lab
Can a model group colours without being told what a flower is? By the end, you will have learned three palettes and measured how stable one of them is when the lighting changes.

## Set up your environment
Open this notebook in JupyterLab or VS Code with a Python kernel. If needed, run `%pip install numpy matplotlib pillow scikit-learn` in a notebook cell, then restart the kernel. The sample photograph is bundled with scikit-learn. Once the packages are installed, you can run the lab offline on a CPU.

Before running the cells, write down one detail you expect to lose with only two colours.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

image_path = None  # Optional: set this to the path of your own photograph.
if image_path is None:
    photo = Image.fromarray(load_sample_image("flower.jpg"))
else:
    with Image.open(Path(image_path)) as source:
        photo = source.convert("RGB")

photo.thumbnail((480, 480))
rgb = np.asarray(photo.convert("RGB"))
pixels = rgb.reshape(-1, 3).astype(np.float32)
print(f"Image shape: {rgb.shape}. Feature table: {pixels.shape}.")
print("First pixel [red, green, blue]:", pixels[0])
plt.imshow(rgb)
plt.axis("off")
plt.show()

## Start with a rule you can inspect
Highlight pixels where green strictly exceeds both other channels. Look for anything highlighted that you would not call a leaf. This rule has no fitting step.

In [ ]:
red, green, blue = rgb.transpose(2, 0, 1)
green_dominant = (green > red) & (green > blue)
plt.imshow(green_dominant, cmap="gray", vmin=0, vmax=1)
plt.title("White = green channel is strongest")
plt.axis("off")
plt.show()

## Learn a palette from the data
Each row contains three features. The model fits centres to a reproducible sample of up to 5,000 pixels, then assigns every pixel to a centre. Compare the results on the same image. Which part of your prediction was accurate?

In [ ]:
rng = np.random.default_rng(42)
fit_rows = rng.choice(len(pixels), size=min(5000, len(pixels)), replace=False)
training_pixels = pixels[fit_rows]
if np.unique(training_pixels, axis=0).shape[0] < 8:
    raise ValueError("Choose a photograph with at least eight distinct colours in the sample.")

models = {}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for k, axis in zip((2, 4, 8), axes):
    model = KMeans(n_clusters=k, n_init=5, random_state=42).fit(training_pixels)
    models[k] = model
    labels = model.predict(pixels)
    reconstructed = model.cluster_centers_[labels].reshape(rgb.shape)
    axis.imshow(reconstructed.round().astype(np.uint8))
    axis.set_title(f"K = {k}")
    axis.axis("off")
plt.tight_layout()
plt.show()

## Change the light while keeping the model fixed
Predict what will happen when you multiply every channel by 0.65. Reuse the fitted K = 4 model so that group numbers mean the same thing in both predictions. White in the change map marks pixels assigned to a different centre.

In [ ]:
model = models[4]
darker = np.rint(rgb * 0.65).astype(np.uint8)
original_labels = model.predict(pixels)
darker_labels = model.predict(darker.reshape(-1, 3).astype(np.float32))
changed = (original_labels != darker_labels).reshape(rgb.shape[:2])
print(f"Pixels assigned to a different group: {100 * changed.mean():.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, image, title in zip(axes, (rgb, darker, changed), ("Original", "Reduced light", "Changed group")):
    if image.ndim == 2:
        axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    else:
        axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## Your evidence
Edit this cell before submitting.

1. My initial prediction and my observations at K = 2, 4 and 8:
2. One example in the feature table is … Its features are … The target labels supplied were …
3. The percentage that changed was … One region of the image that helps explain this result is …
4. Similar colour fails to identify an object when …
5. A non-image task where grouping is useful is … One observation would be … Its two features would be …